# Hybrid 1D-CNN + BiLSTM for Wearable Stress Recognition — WESAD
**OSEMN: Obtain → Scrub → Explore → Model → iNterpret**

One hybrid deep learning model learns directly from BVP and EDA waveforms.
The presentation contract is retained: a completed 30-second window, a 5-second
update stride, and Baseline / Stress / Amusement outputs. There are no classical
classifiers, handcrafted HRV predictors, or model-comparison exports.

**Only one file is saved to Kaggle working: `stress_app_bundle.zip`.** It contains
`stress_model.keras`, `model_config.json`, `stress_inference.py`, and `results.html`.
Figures stay inside the notebook or the single HTML report. Deterministic robust
normalization lives in the shared runtime; learned normalization is inside the model.

The saved model is refitted on all usable participants for app development.
Performance comes from separate participant-held-out predictions, not from that
final model's training score. The >80% macro-F1 target is tested, never assumed.


## 1. Obtain — configuration and synchronized wrist data
Attach the WESAD Kaggle dataset, choose a TensorFlow GPU environment, and Run All.
No neural-enable switch is necessary. The source notebook is self-contained.
scikit-learn is used only for participant splits, class weights and metrics;
TensorFlow/Keras trains the only model. Use trusted WESAD pickle files only.


In [ ]:
from pathlib import Path
from dataclasses import dataclass
from collections import Counter
import base64
import csv
import html
import io
import json
import os
import pickle
import platform
import tempfile
import types
import zipfile
import hashlib
import importlib.metadata

import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from sklearn.model_selection import LeaveOneGroupOut, StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             roc_auc_score, confusion_matrix, classification_report)

SEED = 42
DATA_ROOT = Path(os.environ.get('WESAD_DATA_ROOT',
    '/kaggle/input/wesad-wearable-stress-affect-detection-dataset/WESAD/'))
OUTPUT = Path(os.environ.get('WESAD_OUTPUT_DIR', '/kaggle/working' if Path('/kaggle').exists() else 'hybrid_outputs'))
OUTPUT.mkdir(parents=True, exist_ok=True)
CLASS_NAMES = ['Baseline', 'Stress', 'Amusement']
CLASSES, LABEL_MAP = np.arange(3), {1: 0, 2: 1, 3: 2}
FS, LABEL_FS = {'BVP': 64., 'EDA': 4.}, 700.
MAX_EPOCHS, BATCH_SIZE, PATIENCE = 50, 64, 10
LEARNING_RATES, INNER_SPLITS = [3e-4], 3
EDA_CONTEXT_OPTIONS = [True, False]  # validation chooses whether absolute EDA level helps
OUTER_INNER_SPLITS, EARLY_START_EPOCH = 2, 6
SENSOR_PERMUTATION_REPEATS = 3
EXPERIMENT_LABEL = 'WESAD participant-held-out experiment'  # validation harness marks synthetic tests explicitly
HISTORICAL_REFERENCE = {'source': 'User-provided real WESAD v1 results bundle',
    'subject_macro_f1_mean': 0.4075316510737224, 'pooled_macro_f1': 0.5811528551171493,
    'pure_target_windows': 6360, 'accepted_windows': 2666, 'pulse_rejected_windows': 3694,
    'accepted_class_counts': {'Baseline': 1823, 'Stress': 285, 'Amusement': 558},
    'note': 'Descriptive historical reference; revised retention changes the test population.'}
tf.keras.utils.set_random_seed(SEED)
for gpu in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass
versions = {'python': platform.python_version(), 'tensorflow': tf.__version__}
for package in ['numpy', 'pandas', 'scipy', 'scikit-learn', 'matplotlib']:
    versions[package] = importlib.metadata.version(package)
print('Hybrid CNN–BiLSTM | GPU:', bool(tf.config.list_physical_devices('GPU')))
print('Final download:', OUTPUT / 'stress_app_bundle.zip')


### Parse E4 files and verify their alignment
Uniform E4 files have an epoch row, a sample-rate row, then samples. Malformed
sample rows retain their time position as NaN. Both extracted E4 directories and
subject zip archives are supported. All available folders S2–S17 are considered.

The pickle's synchronized wrist arrays and 700 Hz labels define the shared origin.
Two separated templates must agree before raw E4 samples replace a synchronized
array. An epoch header alone cannot locate the label origin. Unverified raw files
fall back to synchronized wrist arrays. Questionnaires remain audit metadata.
BVP and EDA are both required; unusable participants are reported and skipped.


In [ ]:
@dataclass
class E4Stream:
    epoch: float
    fs: float
    values: np.ndarray

def read_e4_text(subject_dir, name):
    direct = subject_dir / f'{subject_dir.name}_E4_Data' / name
    if direct.is_file():
        return direct.read_text(encoding='utf-8-sig')
    archive = subject_dir / f'{subject_dir.name}_E4_Data.zip'
    if archive.is_file():
        with zipfile.ZipFile(archive) as z:
            matches = [p for p in z.namelist() if Path(p).name.lower() == name.lower()]
            if len(matches) == 1:
                return z.read(matches[0]).decode('utf-8-sig')
    raise FileNotFoundError(str(direct))

def read_uniform_e4(text, expected_fs):
    rows = list(csv.reader(io.StringIO(text)))
    if len(rows) < 3:
        raise ValueError('E4 file has no signal samples')
    epoch, fs = float(rows[0][0]), float(rows[1][0])
    if not np.isfinite(epoch) or not 0 < fs <= 1000:
        raise ValueError('Invalid E4 epoch or sampling rate')
    if not np.isclose(fs, expected_fs):
        raise ValueError(f'Expected {expected_fs} Hz, found {fs} Hz')
    values = np.full(len(rows) - 2, np.nan)
    for i, row in enumerate(rows[2:]):
        try:
            values[i] = float(row[0])
        except (ValueError, IndexError):
            pass
    values[~np.isfinite(values)] = np.nan
    return E4Stream(epoch, fs, values)

def read_questionnaire(path):
    if not path.exists():
        return []
    text = path.read_text(encoding='utf-8-sig', errors='replace')
    # Preserve ragged protocol/self-report rows without guessing their semantics.
    delimiter = ';' if text.count(';') > text.count(',') else ','
    return list(csv.reader(io.StringIO(text), delimiter=delimiter))

def discover_subjects(root):
    if not root.is_dir():
        raise FileNotFoundError(f'WESAD directory not found: {root}. Attach the dataset or set WESAD_DATA_ROOT.')
    audit, subjects = [], []
    for number in range(2, 18):
        folder = root / f'S{number}'
        exists = folder.is_dir() and (folder / f'S{number}.pkl').is_file()
        audit.append({'subject': folder.name, 'discovery': 'available' if exists else 'missing folder/pickle'})
        if exists:
            subjects.append(folder)
    if not subjects:
        raise RuntimeError('No valid S2–S17 participant folders contain synchronized pickle files.')
    return subjects, pd.DataFrame(audit)

def verified_trim_offset(raw, synced, fs):
    """Match two templates; return raw sample index corresponding to synced t=0."""
    raw, synced = np.asarray(raw, float), np.asarray(synced, float)
    if len(raw) < len(synced) or len(synced) < int(45 * fs):
        return None
    length = min(int(20 * fs), len(synced) // 4)
    offsets = []
    for fraction in (.2, .7):
        start = int(fraction * (len(synced) - length))
        template = synced[start:start + length]
        if not np.isfinite(template).all() or np.std(template) < 1e-8:
            return None
        centered = template - template.mean()
        filled = np.nan_to_num(raw)
        numerator = signal.correlate(filled, centered, mode='valid', method='fft')
        sums = np.r_[0., np.cumsum(filled)]
        sums2 = np.r_[0., np.cumsum(filled * filled)]
        variance = sums2[length:] - sums2[:-length] - (sums[length:] - sums[:-length])**2 / length
        denom = np.sqrt(np.maximum(variance, 0) * np.sum(centered**2))
        score = numerator / np.maximum(denom, 1e-12)
        location = int(np.argmax(score))
        actual = raw[location:location + length]
        # Verify sample values as well as correlation. Constant/ambiguous matches fail.
        if score[location] < .99999 or not np.allclose(actual, template, rtol=1e-4, atol=1e-5):
            return None
        offsets.append(location - start)
    if offsets[0] != offsets[1] or not 0 <= offsets[0] <= len(raw) - len(synced):
        return None
    return offsets[0]

def load_subject(folder):
    with (folder / f'{folder.name}.pkl').open('rb') as f:
        data = pickle.load(f, encoding='latin1')
    labels = np.asarray(data['label']).reshape(-1)
    if not len(labels) or not np.isfinite(labels).all() or not np.equal(labels, np.floor(labels)).all():
        raise ValueError('Invalid synchronized labels')
    labels = labels.astype(np.int16)
    wrist = data.get('signal', {}).get('wrist', {})
    if not isinstance(wrist, dict):
        raise ValueError('No synchronized wrist dictionary')
    synced, raw, audit, anchors = {}, {}, [], {}
    for name, fs in FS.items():
        if name in wrist:
            values = np.asarray(wrist[name], dtype=float)
            if values.ndim > 1 and values.shape[1:] != (1,):
                raise ValueError(f'{name} must have one channel')
            synced[name] = values.reshape(-1)
        row = {'subject': folder.name, 'sensor': name, 'raw_status': 'missing', 'source': 'missing'}
        try:
            raw[name] = read_uniform_e4(read_e4_text(folder, name + '.csv'), fs)
            row.update(raw_status='parsed', raw_epoch=raw[name].epoch, raw_samples=len(raw[name].values))
            if name in synced:
                offset = verified_trim_offset(raw[name].values, synced[name], fs)
                if offset is not None:
                    anchors[name] = raw[name].epoch + offset / fs
                    row.update(trim_samples=offset, verified_label_epoch=anchors[name])
                else:
                    row['raw_status'] = 'parsed; no verified match'
        except (FileNotFoundError, ValueError, OSError, zipfile.BadZipFile) as exc:
            row['raw_status'] = str(exc)
        audit.append(row)
    # BVP supplies the most distinctive template. Other accepted anchors must agree.
    origin = anchors.get('BVP', next(iter(anchors.values()), None))
    streams = {}
    duration = len(labels) / LABEL_FS
    for row in audit:
        name, fs = row['sensor'], FS[row['sensor']]
        if name in synced:
            values = synced[name]
            if abs(len(values) / fs - duration) > 1.:
                raise ValueError(f'{name} and label durations differ by more than 1 s')
            if name in anchors and abs(anchors[name] - origin) <= max(1 / fs, 1 / 64):
                offset = int(row['trim_samples'])
                values = raw[name].values[offset:offset + len(values)]
                row['source'] = 'verified raw E4'
            else:
                row['source'] = 'synchronized pickle wrist'
            streams[name] = values
        elif name in raw and origin is not None:
            # A missing synchronized sensor may use the independently verified E4 epoch.
            st = raw[name]
            times = st.epoch - origin + np.arange(len(st.values)) / fs
            target = np.arange(int(duration * fs)) / fs
            streams[name] = np.interp(target, times, st.values, left=np.nan, right=np.nan)
            row['source'] = 'raw E4 via verified shared epoch'
    if 'BVP' not in streams:
        raise ValueError('No aligned BVP available; cannot extract pulse intervals')
    quest = read_questionnaire(folder / f'{folder.name}_quest.csv')
    return {'subject': folder.name, 'labels': labels, 'streams': streams,
            'audit': audit, 'questionnaire_rows': len(quest)}


subject_dirs, discovery = discover_subjects(DATA_ROOT)
print(discovery.to_string(index=False))


## 2. Scrub — the same signal processing in training and the app
The embedded module below is also the `stress_inference.py` shipped in the bundle.
BVP is filtered at 0.5–4 Hz; EDA is low-pass filtered at 1 Hz. Both use causal
fourth-order Butterworth filters. Short gaps use bounded forward fill; long gaps
reset filters and require another warmup. Keep missing sample positions as NaN.

Both channels share window boundaries while retaining their own sample rates:
BVP `(1920, 1)` uses within-window median/MAD scaling. EDA `(120, 4)` contains
log level, relative log level, relative phasic activity and the log derivative.
A causal exponential smoother approximates tonic activity; this is not a validated
physiological decomposition. No full-test-record statistics or labeled baseline is used.
Missingness and flat-signal rules reject windows. Pulse plausibility is advisory:
the supplied real run lost 58% of pure target windows to that heuristic. The report
also evaluates the old pulse-eligible subset. Retaining noisy pulses may hurt results.

A window is retained only when every label is the same included protocol state.
Original labels 1/2/3 map to 0/1/2; meditation, transitions and other labels are
excluded. Amusement is positive arousal, not a relaxation label. Label purity is an
offline training/evaluation rule and cannot screen unknown transitions in live use.


In [ ]:
INFERENCE_SOURCE = r'''"""Shared training/app preprocessing and Python-backend inference for wrist BVP + EDA.

Load StressPredictor from the extracted bundle. Pass raw, uniformly sampled arrays
from the SAME recording origin: BVP 64 Hz, EDA 4 Hz, missing samples represented by
NaN (never deleted). Supply the recording from session start so filter history is
preserved. This reference backend replays the session on each call; it is not an
optimized BLE client or a browser/TFLite runtime. Do not normalize inputs yourself.
"""
from pathlib import Path
import json
import numpy as np
from scipy import signal

DEFAULT_PREPROCESSING = {
    'version': 2, 'bvp_fs': 64, 'eda_fs': 4, 'window_sec': 30, 'stride_sec': 5,
    'filter_order': 4, 'bvp_band_hz': [.5, 4.], 'eda_lowpass_hz': 1.,
    'warmup_sec': 10., 'max_gap_sec': .5, 'minimum_observed_fraction': .95,
    'eda_bounds_us': [0., 100.], 'minimum_std': 1e-8,
    'peak_distance_sec': .30, 'peak_prominence_mad': .5,
    'ibi_bounds_sec': [.30, 2.], 'ibi_local_tolerance': .25,
    'minimum_intervals': 15, 'minimum_ibi_fraction': .8,
    'minimum_ibi_coverage': .8, 'maximum_beat_gap_sec': 3.,
    'strict_pulse_gate': False, 'tonic_time_constant_sec': 10.,
    'bvp_mad_floor': 1e-6, 'eda_log_mad_floor': .02,
    'relative_tonic_floor_us': .05, 'robust_clip': 5.,
    'eda_channels': ['log_level', 'relative_log_level', 'relative_phasic', 'log_derivative']
}

def finite_runs(mask):
    edges = np.diff(np.r_[False, mask, False].astype(np.int8))
    return zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1))

def clean_channel(values, fs, config, bounds=None):
    x = np.asarray(values, dtype=float)
    if x.ndim == 2 and x.shape[1] == 1:
        x = x[:, 0]
    if x.ndim != 1:
        raise ValueError('Each signal must be a single channel.')
    x = x.copy()
    x[~np.isfinite(x)] = np.nan
    if bounds is not None:
        x[(x < bounds[0]) | (x > bounds[1])] = np.nan
    observed = np.isfinite(x)
    limit = int(config['max_gap_sec'] * fs)
    for a, b in finite_runs(~observed):
        if a > 0:
            x[a:min(b, a + limit)] = x[a - 1]
    return x, observed

def filter_channel(x, fs, cutoff, kind, config):
    sos = signal.butter(config['filter_order'], cutoff, btype=kind, fs=fs, output='sos')
    result = np.full(len(x), np.nan)
    warmup = int(config['warmup_sec'] * fs)
    for a, b in finite_runs(np.isfinite(x)):
        if b - a <= warmup:
            continue
        result[a:b], _ = signal.sosfilt(sos, x[a:b], zi=signal.sosfilt_zi(sos) * x[a])
        result[a:a + warmup] = np.nan
    return result

def prepare_recording(bvp, eda, config=None):
    config = DEFAULT_PREPROCESSING if config is None else config
    bvp_cleaned, bv = clean_channel(bvp, config['bvp_fs'], config)
    e, ev = clean_channel(eda, config['eda_fs'], config, config['eda_bounds_us'])
    ef = filter_channel(e, config['eda_fs'], config['eda_lowpass_hz'], 'lowpass', config)
    tonic, derivative = np.full(len(ef), np.nan), np.full(len(ef), np.nan)
    alpha = 1 - np.exp(-1 / (config['eda_fs'] * config['tonic_time_constant_sec']))
    for a, b in finite_runs(np.isfinite(ef)):
        positive = np.maximum(ef[a:b], 0.)
        tonic[a:b], _ = signal.lfilter([alpha], [1., -(1-alpha)], positive,
                                       zi=[(1-alpha) * positive[0]])
        derivative[a:b] = np.diff(np.log1p(positive), prepend=np.log1p(positive[0])) * config['eda_fs']
    return {'bvp': filter_channel(bvp_cleaned, config['bvp_fs'], config['bvp_band_hz'], 'bandpass', config),
            'eda': ef, 'eda_tonic': tonic, 'eda_phasic': ef - tonic, 'eda_log_derivative': derivative,
            'bvp_observed': bv, 'eda_observed': ev}

def robust_scale(x, floor, clip):
    median = np.median(x)
    scale = max(float(1.4826 * np.median(np.abs(x - median))), floor)
    return np.clip((x - median) / scale, -clip, clip)

def represent_window(bvp, eda, tonic, derivative, config):
    # Only the completed window supplies robust statistics: no future samples or labels.
    bvp = robust_scale(bvp, config['bvp_mad_floor'], config['robust_clip'])[:, None]
    log_level = np.log1p(np.maximum(eda, 0.))
    relative = robust_scale(log_level, config['eda_log_mad_floor'], config['robust_clip'])
    phasic = (eda - tonic) / (np.maximum(tonic, 0.) + config['relative_tonic_floor_us'])
    channels = np.column_stack([log_level, relative,
                                np.clip(phasic, -config['robust_clip'], config['robust_clip']),
                                np.clip(derivative, -config['robust_clip'], config['robust_clip'])])
    return {'bvp': bvp.astype('float32'), 'eda': channels.astype('float32')}

def pulse_quality(bvp, config):
    mad = np.median(np.abs(bvp - np.median(bvp)))
    peaks, _ = signal.find_peaks(bvp, distance=max(1, int(config['peak_distance_sec'] * config['bvp_fs'])),
                                prominence=max(config['peak_prominence_mad'] * mad, 1e-6))
    rr = np.diff(peaks) / config['bvp_fs']
    if len(rr) < config['minimum_intervals']:
        return False
    local = np.array([np.median(rr[max(0, i-5):i+6]) for i in range(len(rr))])
    good = ((rr >= config['ibi_bounds_sec'][0]) & (rr <= config['ibi_bounds_sec'][1]) &
            (np.abs(rr - local) <= config['ibi_local_tolerance'] * local))
    beat_times = peaks[1:][good] / config['bvp_fs']
    return bool(good.sum() >= config['minimum_intervals'] and
                good.mean() >= config['minimum_ibi_fraction'] and
                rr[good].sum() / config['window_sec'] >= config['minimum_ibi_coverage'] and
                len(beat_times) > 1 and np.max(np.diff(beat_times)) <= config['maximum_beat_gap_sec'])

def window_at(prepared, end_sec, config=None):
    config = DEFAULT_PREPROCESSING if config is None else config
    start = end_sec - config['window_sec']
    if start < 0:
        return None, 'insufficient_history'
    inputs = {}
    for name in ['bvp', 'eda']:
        fs = config[name + '_fs']
        a, b = int(round(start * fs)), int(round(end_sec * fs))
        x = prepared[name][a:b]
        if len(x) != int(config['window_sec'] * fs):
            return None, name + '_insufficient_samples'
        if not np.isfinite(x).all():
            return None, name + '_gap_or_warmup'
        if prepared[name + '_observed'][a:b].mean() < config['minimum_observed_fraction']:
            return None, name + '_missing_samples'
        if np.std(x) < config['minimum_std']:
            return None, name + '_flat'
        inputs[name] = x.astype('float32')[:, None]
    if config['strict_pulse_gate'] and not pulse_quality(inputs['bvp'][:, 0], config):
        return None, 'pulse_quality'
    a = int(round(start * config['eda_fs']))
    b = int(round(end_sec * config['eda_fs']))
    tonic, derivative = prepared['eda_tonic'][a:b], prepared['eda_log_derivative'][a:b]
    if not np.isfinite(tonic).all() or not np.isfinite(derivative).all():
        return None, 'eda_component_gap'
    return represent_window(inputs['bvp'][:, 0], inputs['eda'][:, 0], tonic, derivative, config), 'ok'

class StressPredictor:
    """Load once per backend process; call predict_latest with a complete raw session."""
    def __init__(self, bundle_directory):
        import tensorflow as tf
        folder = Path(bundle_directory)
        self.config = json.loads((folder / 'model_config.json').read_text(encoding='utf-8'))
        if self.config.get('format_version') != 2 or self.config.get('preprocessing', {}).get('version') != 2:
            raise ValueError('Use the version 2 model, config and inference module from the same bundle.')
        self.preprocessing = self.config['preprocessing']
        self.model = tf.keras.models.load_model(folder / 'stress_model.keras', compile=False)
        self.labels = self.config['class_names']

    def predict_latest(self, bvp, eda):
        c = self.preprocessing
        prepared = prepare_recording(bvp, eda, c)
        duration = min(len(prepared['bvp']) / c['bvp_fs'], len(prepared['eda']) / c['eda_fs'])
        end = np.floor(duration / c['stride_sec']) * c['stride_sec']
        inputs, status = window_at(prepared, end, c)
        if inputs is None:
            return {'status': status, 'window_end_sec': float(end), 'prediction': None,
                    'probabilities': None}
        probabilities = self.model({k: v[None, ...] for k, v in inputs.items()}, training=False).numpy()[0]
        index = int(np.argmax(probabilities))
        a, b = int((end-c['window_sec'])*c['bvp_fs']), int(end*c['bvp_fs'])
        warning = None if pulse_quality(prepared['bvp'][a:b], c) else 'irregular_pulse_pattern'
        return {'status': 'ok', 'window_start_sec': float(end - c['window_sec']),
                'window_end_sec': float(end), 'class_id': index, 'prediction': self.labels[index],
                'quality_warning': warning,
                'probabilities': {label: float(p) for label, p in zip(self.labels, probabilities)}}
'''
runtime = types.ModuleType('stress_runtime')
exec(compile(INFERENCE_SOURCE, 'stress_inference.py', 'exec'), runtime.__dict__)
PREPROCESSING = dict(runtime.DEFAULT_PREPROCESSING)

def target_at(labels, start, end):
    a, b = int(round(start * LABEL_FS)), int(round(end * LABEL_FS))
    if a < 0 or b > len(labels) or b <= a:
        return None
    chunk = labels[a:b]
    original = int(chunk[0])
    return LABEL_MAP[original] if original in LABEL_MAP and np.all(chunk == original) else None

eda_exploration_rows, class_retention_rows = [], []

def subject_windows(record):
    streams = record['streams']
    if 'BVP' not in streams or 'EDA' not in streams:
        raise ValueError('Both aligned BVP and EDA are required for the hybrid model.')
    prepared = runtime.prepare_recording(streams['BVP'], streams['EDA'], PREPROCESSING)
    duration = min(len(record['labels']) / LABEL_FS,
                   len(streams['BVP']) / FS['BVP'], len(streams['EDA']) / FS['EDA'])
    bb, ee, rows, audit, class_counts = [], [], [], Counter(), {k: Counter() for k in CLASSES}
    for end in np.arange(PREPROCESSING['window_sec'], duration + 1e-7, PREPROCESSING['stride_sec']):
        start = end - PREPROCESSING['window_sec']
        audit['candidate'] += 1
        target = target_at(record['labels'], start, end)
        if target is None:
            audit['excluded_or_mixed_label'] += 1
            continue
        class_counts[target]['pure_candidates'] += 1
        ea, eb = int(start * FS['EDA']), int(end * FS['EDA'])
        ex = prepared['eda'][ea:eb]
        finite = ex[np.isfinite(ex)]
        # Explore every pure target window before pulse-based screening can alter class balance.
        if len(finite) >= .95 * len(ex):
            ph = prepared['eda_phasic'][ea:eb]
            eda_exploration_rows.append({'subject': record['subject'], 'target': target,
                'eda_median_us': float(np.median(finite)),
                'eda_log_median': float(np.median(np.log1p(np.maximum(finite, 0)))),
                'phasic_std_us': float(np.nanstd(ph)),
                'eda_slope_us_per_min': float(np.polyfit(np.flatnonzero(np.isfinite(ex))/FS['EDA'], finite, 1)[0]*60)})
        inputs, status = runtime.window_at(prepared, end, PREPROCESSING)
        if inputs is None:
            audit[status] += 1
            class_counts[target][status] += 1
            continue
        ba, be = int(start * FS['BVP']), int(end * FS['BVP'])
        legacy_eligible = runtime.pulse_quality(prepared['bvp'][ba:be], PREPROCESSING)
        class_counts[target]['accepted'] += 1
        class_counts[target]['legacy_accepted'] += int(legacy_eligible)
        audit['pulse_advisory'] += int(not legacy_eligible)
        bb.append(inputs['bvp'])
        ee.append(inputs['eda'])
        rows.append({'subject': record['subject'], 'start_sec': start, 'end_sec': end, 'target': target,
                     'legacy_eligible': legacy_eligible, 'window_id': f"{record['subject']}:{start:.3f}"})
        audit['accepted'] += 1
    for k, counts in class_counts.items():
        class_retention_rows.append({'subject': record['subject'], 'target': int(k),
            'state': CLASS_NAMES[k], 'pure_candidates': counts['pure_candidates'],
            'accepted': counts['accepted'], 'legacy_accepted': counts['legacy_accepted'],
            'rejected_reasons': {key: value for key, value in counts.items()
                                 if key not in ['pure_candidates','accepted','legacy_accepted']}})
    return bb, ee, rows, dict(audit), prepared


### Build model inputs in memory
Only deterministic processing occurs here. Normalization statistics, class weights,
stopping epochs and learning rates are learned later within training participants.
No raw recordings, window arrays, fold models or intermediate tables are exported.


In [ ]:
bvp_windows, eda_windows, window_rows = [], [], []
ingestion_audit, alignment_audit, errors = [], [], []
example = None
for folder in subject_dirs:
    try:
        record = load_subject(folder)
        bb, ee, rows, audit, prepared = subject_windows(record)
        bvp_windows.extend(bb)
        eda_windows.extend(ee)
        window_rows.extend(rows)
        ingestion_audit.append({'subject': folder.name, **audit, 'questionnaire_rows': record['questionnaire_rows']})
        alignment_audit.extend(record['audit'])
        if example is None and rows:
            example = {'subject': folder.name, 'streams': record['streams'], 'prepared': prepared,
                       'start_sec': rows[0]['start_sec']}
        print(f"{folder.name}: {len(rows)} accepted / {audit.get('candidate', 0)} candidate windows")
    except (OSError, ValueError, KeyError, TypeError, pickle.UnpicklingError, EOFError) as exc:
        errors.append({'subject': folder.name, 'error': str(exc)})
        print(f'SKIPPED {folder.name}: {exc}')
    finally:
        for variable in ['record', 'prepared', 'bb', 'ee']:
            globals().pop(variable, None)
if not window_rows:
    raise RuntimeError('No usable BVP/EDA windows. Inspect the participant messages above.')
NB, NE = np.stack(bvp_windows), np.stack(eda_windows)
del bvp_windows, eda_windows
meta = pd.DataFrame(window_rows)
y, groups = meta.target.to_numpy(dtype=int), meta.subject.to_numpy()
assert NB.shape[1:] == (1920, 1) and NE.shape[1:] == (120, 4)
assert not meta.window_id.duplicated().any()
if len(np.unique(groups)) < 4 or set(y) != set(CLASSES):
    raise RuntimeError('Nested evaluation needs at least four usable participants and all three classes.')
print('Input shapes:', NB.shape, NE.shape)


## 3. Explore — waveform quality and class balance
These displays remain in the notebook. They describe the cohort and do not select
model settings. All retained windows are split by participant during evaluation.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 7))
for axis, name, seconds in zip(axes, ['BVP', 'EDA'], [15, 30]):
    start = example['start_sec']
    a, b = int(start * FS[name]), int((start + seconds) * FS[name])
    times = np.arange(a, b) / FS[name]
    axis.plot(times, example['streams'][name][a:b], alpha=.5, label='Raw')
    axis.plot(times, example['prepared'][name.lower()][a:b], label='Causal filtered')
    axis.set(title=f"{example['subject']}: {name}", xlabel='Seconds from synchronized origin',
             ylabel='BVP (a.u.)' if name == 'BVP' else 'EDA (µS)')
    axis.legend()
ea, eb = int(start * FS['EDA']), int((start + 30) * FS['EDA'])
et = np.arange(ea, eb) / FS['EDA']
axes[1].plot(et, example['prepared']['eda_tonic'][ea:eb], label='Approximate tonic (causal EMA)', ls='--')
axes[1].legend()
axes[2].plot(et, example['prepared']['eda_phasic'][ea:eb], color='#d89e36')
axes[2].axhline(0, color='grey', lw=.8)
axes[2].set(title='EDA phasic residual (approximation)', xlabel='Seconds', ylabel='µS')
plt.tight_layout()
plt.show()
plt.close(fig)
del example
balance = pd.crosstab(meta.subject, meta.target).reindex(columns=CLASSES, fill_value=0)
balance.columns = CLASS_NAMES
balance.plot.bar(stacked=True, figsize=(11, 4), title='Accepted windows by participant and class')
plt.ylabel('Windows')
plt.tight_layout()
plt.show()
plt.close()

eda_cohort = pd.DataFrame(eda_exploration_rows).groupby(['subject','target'], as_index=False).median(numeric_only=True)
retention = pd.DataFrame(class_retention_rows)
retention['retention_pct'] = 100 * retention.accepted / retention.pure_candidates.replace(0, np.nan)
retention['legacy_retention_pct'] = 100 * retention.legacy_accepted / retention.pure_candidates.replace(0, np.nan)
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for axis, feature, title in [(axes[0,0], 'eda_log_median', 'EDA level: subject/class medians before quality rejection'),
                              (axes[0,1], 'phasic_std_us', 'EDA response variation: subject/class medians')]:
    data = [eda_cohort.loc[eda_cohort.target == k, feature].dropna().to_numpy() for k in CLASSES]
    axis.boxplot(data, labels=CLASS_NAMES)
    for k, values in enumerate(data):
        axis.scatter(np.full(len(values), k+1), values, s=15, alpha=.6)
    axis.set(title=title, ylabel=feature)
retention_matrix = retention.pivot(index='subject', columns='state', values='retention_pct').reindex(columns=CLASS_NAMES)
im = axes[1,0].imshow(retention_matrix.to_numpy(), vmin=0, vmax=100, aspect='auto', cmap='Blues')
axes[1,0].set(xticks=np.arange(3), xticklabels=CLASS_NAMES, yticks=np.arange(len(retention_matrix)),
              yticklabels=retention_matrix.index, title='New retention of pure target windows (%)')
for i in range(len(retention_matrix)):
    for j in range(3):
        value = retention_matrix.iloc[i,j]
        axes[1,0].text(j, i, f'{value:.0f}' if np.isfinite(value) else 'n/a', ha='center', va='center',
                       color='white' if value > 60 else 'black', fontsize=8)
totals = retention.groupby('target')[['pure_candidates','accepted','legacy_accepted']].sum().reindex(CLASSES, fill_value=0)
for offset, key, label in [(-.25,'pure_candidates','Pure candidates'),(0,'legacy_accepted','Old strict gate'),(.25,'accepted','New waveform QC')]:
    axes[1,1].bar(np.arange(3)+offset, totals[key], width=.25, label=label)
axes[1,1].set(xticks=np.arange(3), xticklabels=CLASS_NAMES, title='Where class imbalance enters', ylabel='Windows')
axes[1,1].legend()
fig.suptitle('EDA exploration and class-specific retention — ' + EXPERIMENT_LABEL)
plt.tight_layout()
eda_buffer = io.BytesIO()
fig.savefig(eda_buffer, format='png', dpi=140, bbox_inches='tight')
eda_plot_data = base64.b64encode(eda_buffer.getvalue()).decode('ascii')
plt.show()
plt.close(fig)
print('Retention by class:\n', totals.to_string())
missing_cells = retention[(retention.pure_candidates > 0) & (retention.accepted == 0)]
if len(missing_cells):
    print('WARNING: classes absent after screening:', missing_cells[['subject','state']].to_dict(orient='records'))


## 4. Model — one hybrid 1D-CNN + BiLSTM
Each branch learns local waveform patterns. BVP pooling reduces 1920 samples to
30 time steps with 64 channels; EDA pooling reduces 120 samples to 30 steps with
32 channels. Their concatenation is `(30, 96)`, followed by a BiLSTM with 32 units
per direction, attention and mean pooling, Dense(32), dropout(0.4), and softmax.
Residual/dilated convolutions learn patterns at multiple time scales.

The shared runtime constructs robust inputs; embedded channel normalization uses
training participants only. Inner subject validation selects whether absolute EDA
level helps. Participant/class weights reduce domination by large baseline groups.
AdamW uses cosine learning-rate decay; stopping selection starts after six epochs.
The BiLSTM and attention see only the completed window, without future windows.
With 15 participants, default tuning performs 82 neural fits including the final
model. Allow a full GPU session; early stopping limits each fit to at most 50 epochs.


In [ ]:
def participant_class_weights(indices):
    # Equal total weight per observed participant/class cell; no synthetic examples.
    cells = pd.DataFrame({'subject': groups[indices], 'target': y[indices]})
    counts = cells.groupby(['subject','target']).target.transform('size').to_numpy()
    weights = 1. / counts
    return (weights / weights.mean()).astype('float32')

def channel_statistics(indices):
    result = {}
    weights = participant_class_weights(indices)
    for name, values in [('bvp', NB), ('eda', NE)]:
        total = np.zeros(values.shape[-1], dtype=float)
        squares, mass = total.copy(), 0.
        for a in range(0, len(indices), 128):
            block = values[indices[a:a+128]].astype('float64')
            w = weights[a:a+128, None, None]
            total += (block*w).sum(axis=(0,1))
            squares += (block*block*w).sum(axis=(0,1))
            mass += float(w.sum()) * block.shape[1]
        mean = total / mass
        result[name] = {'mean': mean.tolist(), 'variance': np.maximum(squares/mass-mean**2, 1e-6).tolist()}
    return result

def build_model(indices, learning_rate, absolute_eda=True):
    layers = tf.keras.layers
    statistics = channel_statistics(indices)
    bvp = layers.Input((1920, 1), name='bvp')
    eda = layers.Input((120, 4), name='eda')
    b = layers.Normalization(axis=-1, name='bvp_normalization', **statistics['bvp'])(bvp)
    e = layers.Normalization(axis=-1, name='eda_normalization', **statistics['eda'])(eda)
    if not absolute_eda:
        # Serializable built-in layers omit only channel 0, keeping the public 4-channel input contract.
        e = layers.Permute((2,1))(e)
        e = layers.Cropping1D(cropping=(1,0))(e)
        e = layers.Permute((2,1))(e)
    def residual_block(x, channels, kernel, pool):
        shortcut = layers.Conv1D(channels, 1, padding='same')(x)
        z = layers.Conv1D(channels, kernel, padding='same', use_bias=False)(x)
        z = layers.LayerNormalization(axis=-1)(z)
        z = layers.Activation('swish')(z)
        z = layers.Conv1D(channels, kernel, dilation_rate=2, padding='same', use_bias=False)(z)
        z = layers.LayerNormalization(axis=-1)(z)
        z = layers.Add()([shortcut, z])
        z = layers.Activation('swish')(z)
        return layers.AveragePooling1D(pool)(z)
    for channels, kernel in [(16,9),(32,5),(64,3)]:
        b = residual_block(b, channels, kernel, 4)
    e = residual_block(e, 16, 5, 2)
    e = residual_block(e, 32, 3, 2)
    fused = layers.Concatenate(name='fusion')([b, e])
    assert tuple(fused.shape[1:]) == (30, 96)
    z = layers.LayerNormalization(axis=-1)(fused)
    z = layers.SpatialDropout1D(.15)(z)
    z = layers.Bidirectional(layers.LSTM(32, return_sequences=True), name='bilstm')(z)
    attention = layers.Dense(1, name='attention_logits')(z)
    attention = layers.Softmax(axis=1, name='temporal_attention')(attention)
    context = layers.Dot(axes=1)([attention, z])
    context = layers.Flatten()(context)
    context = layers.Concatenate()([context, layers.GlobalAveragePooling1D()(z)])
    z = layers.Dense(32, activation='swish')(context)
    z = layers.Dropout(.4)(z)
    output = layers.Dense(3, activation='softmax', name='state_probabilities')(z)
    model = tf.keras.Model({'bvp': bvp, 'eda': eda}, output, name='stress_cnn_bilstm')
    schedule = tf.keras.optimizers.schedules.CosineDecay(learning_rate,
        decay_steps=max(1, int(np.ceil(len(indices)/BATCH_SIZE)) * MAX_EPOCHS), alpha=.1)
    model.compile(optimizer=tf.keras.optimizers.AdamW(schedule, weight_decay=1e-4, global_clipnorm=1.),
                  loss='sparse_categorical_crossentropy')
    return model

def inputs_for(indices):
    return {'bvp': NB[indices], 'eda': NE[indices]}

def predict_scores(model, inputs):
    # Direct batched calls also avoid creating a new tf.data thread pool for each score.
    n = len(inputs['bvp'])
    return np.concatenate([model({name: values[a:a+BATCH_SIZE] for name, values in inputs.items()},
                                training=False).numpy() for a in range(0, n, BATCH_SIZE)])

def participant_splits(indices, seed):
    cv = StratifiedGroupKFold(n_splits=min(INNER_SPLITS, len(np.unique(groups[indices]))),
                              shuffle=True, random_state=seed)
    result = []
    for a, b in cv.split(NB[indices, 0, 0], y[indices], groups[indices]):
        train, valid = indices[a], indices[b]
        assert not set(groups[train]) & set(groups[valid])
        if set(y[train]) != set(CLASSES):
            raise ValueError('A training participant split is missing a class. More usable subjects are needed.')
        result.append((train, valid))
    return result

class ValidationMacroF1(tf.keras.callbacks.Callback):
    def __init__(self, indices):
        super().__init__()
        self.inputs, self.truth, self.subjects = inputs_for(indices), y[indices], groups[indices]
    def on_epoch_end(self, epoch, logs=None):
        probabilities = predict_scores(self.model, self.inputs)
        prediction = probabilities.argmax(1)
        logs['val_macro_f1'] = float(np.mean([f1_score(self.truth[self.subjects==subject],
            prediction[self.subjects==subject], labels=CLASSES, average='macro', zero_division=0)
            for subject in np.unique(self.subjects)]))

def tune_hybrid(splits, seed):
    trials = []
    for lr, absolute_eda in [(lr, absolute) for lr in LEARNING_RATES for absolute in EDA_CONTEXT_OPTIONS]:
        scores, epochs = [], []
        for split_number, (train, valid) in enumerate(splits):
            print(f"Inner fit {split_number+1}/{len(splits)} | absolute EDA {absolute_eda} | "
                  f"lr {lr:g} | {len(train)} train / {len(valid)} validation windows", flush=True)
            tf.keras.backend.clear_session()
            tf.keras.utils.set_random_seed(seed + split_number)
            model = build_model(train, lr, absolute_eda)
            history = model.fit(inputs_for(train), y[train],
                sample_weight=participant_class_weights(train),
                validation_data=(inputs_for(valid), y[valid]), epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
                callbacks=[ValidationMacroF1(valid), tf.keras.callbacks.EarlyStopping(
                    monitor='val_macro_f1', mode='max', patience=PATIENCE, restore_best_weights=True,
                    start_from_epoch=min(EARLY_START_EPOCH, MAX_EPOCHS-1))], verbose=0)
            validation = history.history['val_macro_f1']
            first = min(EARLY_START_EPOCH, len(validation)-1)
            best_epoch = first + int(np.argmax(validation[first:])) + 1
            scores.append(float(validation[best_epoch-1]))
            epochs.append(best_epoch)
        trials.append({'learning_rate': lr, 'absolute_eda': absolute_eda, 'validation_macro_f1': float(np.mean(scores)),
                       'epochs': max(1, int(np.median(epochs))), 'split_epochs': epochs,
                       'split_scores': scores})
    return max(trials, key=lambda item: item['validation_macro_f1']), trials

def refit_hybrid(indices, choice, seed):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    model = build_model(indices, choice['learning_rate'], choice['absolute_eda'])
    model.fit(inputs_for(indices), y[indices], sample_weight=participant_class_weights(indices),
              epochs=choice['epochs'], batch_size=BATCH_SIZE, verbose=0)
    return model

def metrics(truth, scores):
    pred = scores.argmax(1)
    observed = np.unique(truth)
    result = {'accuracy': float(accuracy_score(truth, pred)),
              'classes_present': [CLASS_NAMES[k] for k in observed],
              'class_support': {CLASS_NAMES[k]: int(np.sum(truth==k)) for k in CLASSES},
              'f1_macro_observed_classes': float(f1_score(truth, pred, labels=observed, average='macro', zero_division=0))}
    for average in ['macro', 'weighted']:
        result['f1_' + average] = float(f1_score(truth, pred, labels=CLASSES, average=average, zero_division=0))
        result['precision_' + average] = float(precision_score(truth, pred, labels=CLASSES, average=average, zero_division=0))
        result['recall_' + average] = float(recall_score(truth, pred, labels=CLASSES, average=average, zero_division=0))
    aucs = []
    for k in CLASSES:
        binary = truth == k
        auc = float(roc_auc_score(binary, scores[:, k])) if len(np.unique(binary)) == 2 else None
        result['roc_auc_' + CLASS_NAMES[k]] = auc
        if auc is not None:
            aucs.append(auc)
    result['roc_auc_ovr_macro'] = float(np.mean(aucs)) if aucs else None
    result['auc_classes_evaluable'] = len(aucs)
    return result


### Unseen-participant evaluation
Outer LOSO holds out each participant for testing. Within the remaining people,
participant validation splits select EDA context and stopping epoch. A fresh
model then trains on all outer-training participants for the selected epoch count.
The held-out participant never fits normalization, weights, or hyperparameters.

For insight, each channel is shuffled across held-out windows while the other is
retained. The macro-F1 decrease measures reliance on that input for this model;
artificial channel combinations, correlated physiology and overlapping windows
prevent interpreting this diagnostic as a causal biomarker ranking.


In [ ]:
fold_records, sensor_effects = [], []
oof_scores = np.full((len(y), 3), np.nan, dtype='float32')
oof_seen = np.zeros(len(y), dtype=int)
for fold, (train, test) in enumerate(LeaveOneGroupOut().split(NB[:, 0, 0], y, groups)):
    subject = str(groups[test][0])
    assert not set(groups[train]) & set(groups[test])
    if set(y[train]) != set(CLASSES):
        raise ValueError(f'Outer training fold for {subject} is missing a class.')
    inner_splits = participant_splits(train, SEED + fold)[:OUTER_INNER_SPLITS]
    choice, trials = tune_hybrid(inner_splits, SEED + fold)
    model = refit_hybrid(train, choice, SEED + fold)
    test_inputs = inputs_for(test)
    probabilities = predict_scores(model, test_inputs)
    oof_scores[test], oof_seen[test] = probabilities, oof_seen[test] + 1
    measured = metrics(y[test], probabilities)
    legacy_mask = meta.iloc[test].legacy_eligible.to_numpy(dtype=bool)
    legacy_metrics = metrics(y[test][legacy_mask], probabilities[legacy_mask]) if legacy_mask.any() else None
    fold_records.append({'subject': subject, 'windows': len(test), **measured,
                         'learning_rate': choice['learning_rate'], 'epochs': choice['epochs'],
                         'absolute_eda': choice['absolute_eda'], 'legacy_eligible_metrics': legacy_metrics,
                         'legacy_windows': int(legacy_mask.sum()),
                         'inner_validation_subjects': sorted({s for _, valid in inner_splits for s in groups[valid]}),
                         'inner_splits': [{'train': sorted(set(groups[a])), 'validation': sorted(set(groups[b]))}
                                          for a,b in inner_splits],
                         'training_subjects': sorted(set(groups[train])), 'inner_trials': trials})
    rng = np.random.default_rng(SEED + fold)
    for channel in ['bvp', 'eda']:
        drops = []
        for repeat in range(SENSOR_PERMUTATION_REPEATS):
            shuffled = dict(test_inputs)
            shuffled[channel] = test_inputs[channel][rng.permutation(len(test))]
            drops.append(measured['f1_macro'] - metrics(y[test], predict_scores(model, shuffled))['f1_macro'])
        sensor_effects.append({'subject': subject, 'channel': channel, 'macro_f1_decrease': float(np.mean(drops))})
    print(f"LOSO {fold+1}/{len(np.unique(groups))} | {subject} | macro F1 {measured['f1_macro']:.3f}")
    del model
assert np.all(oof_seen == 1) and np.isfinite(oof_scores).all()
assert np.allclose(oof_scores.sum(axis=1), 1., atol=1e-5)


### Refit the final app model on every usable participant
A new participant group-CV search on the full development cohort selects the
final EDA context and median stopping epoch. Then a fresh CNN–BiLSTM, including
its normalization layers, fits every usable participant. No best test fold is
chosen as the deployment model. This final refit has no new independent test score.


In [ ]:
all_indices = np.arange(len(y))
final_choice, final_trials = tune_hybrid(participant_splits(all_indices, SEED + 1000), SEED + 1000)
final_model = refit_hybrid(all_indices, final_choice, SEED + 2000)
print('Final hybrid model trained on', len(np.unique(groups)), 'participants.')
print('Final learning rate:', final_choice['learning_rate'], '| epochs:', final_choice['epochs'])


## 5. iNterpret — one compact report and one app bundle
The primary metric is the mean of held-out participants' macro F1, giving each
person equal weight. Its bootstrap interval resamples participant scores; it is
approximate because LOSO training sets overlap. Pooled metrics and normalized
confusion matrices are secondary because windows overlap.

The report includes metrics, per-class recall, each participant's result, channel
reliance, exclusions, and concise conclusions. Detailed numerical results live in
a collapsible report section instead of a collection of CSV/JSON files.


In [ ]:
folds = pd.DataFrame(fold_records)
subject_f1 = folds.f1_macro.to_numpy()
rng = np.random.default_rng(SEED)
bootstrap = rng.choice(subject_f1, (2000, len(subject_f1)), replace=True).mean(axis=1)
valid_aucs = [row['roc_auc_ovr_macro'] for row in fold_records if row['roc_auc_ovr_macro'] is not None]
summary = {'subject_macro_f1_mean': float(subject_f1.mean()),
           'subject_macro_f1_observed_classes_mean': float(folds.f1_macro_observed_classes.mean()),
           'subject_macro_f1_sd': float(subject_f1.std(ddof=1)),
           'subject_f1_ci95': np.quantile(bootstrap, [.025, .975]).tolist(),
           'subject_accuracy_mean': float(folds.accuracy.mean()),
           'subject_f1_weighted_mean': float(folds.f1_weighted.mean()),
           'subject_roc_auc_ovr_macro_mean': float(np.mean(valid_aucs)) if valid_aucs else None,
           'pooled': metrics(y, oof_scores), 'participants': len(folds), 'accepted_windows': len(y)}
legacy_f1 = [r['legacy_eligible_metrics']['f1_macro'] for r in fold_records if r['legacy_eligible_metrics'] is not None]
legacy_comparison = {'historical_reference': HISTORICAL_REFERENCE,
    'current_model_on_old_gate_subset_subject_macro_f1': float(np.mean(legacy_f1)) if legacy_f1 else None,
    'current_old_gate_windows': int(meta.legacy_eligible.sum()),
    'comparable_retention_counts': int(meta.legacy_eligible.sum()) == HISTORICAL_REFERENCE['accepted_windows'],
    'caution': 'Count agreement alone does not verify identical rows; training populations and preprocessing changed. This is descriptive, not a controlled architecture ablation.'}
per_class = classification_report(y, oof_scores.argmax(1), labels=CLASSES, target_names=CLASS_NAMES,
                                   output_dict=True, zero_division=0)
cm = confusion_matrix(y, oof_scores.argmax(1), labels=CLASSES, normalize='true')
effects = pd.DataFrame(sensor_effects).groupby('channel').macro_f1_decrease.mean()
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
image = axes[0].imshow(cm, vmin=0, vmax=1, cmap='Blues')
for i in range(3):
    for j in range(3):
        axes[0].text(j, i, f'{cm[i, j]:.2f}', ha='center', va='center', color='white' if cm[i, j] > .5 else 'black')
axes[0].set(xticks=range(3), yticks=range(3), xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            xlabel='Predicted', ylabel='Actual', title='Pooled LOSO recall')
axes[1].bar(folds.subject, folds.f1_macro, color='#117b8a')
axes[1].axhline(.8, ls='--', color='grey', label='Presentation target')
axes[1].set(ylim=(0, 1), ylabel='Macro F1', title='Unseen participants')
axes[1].tick_params(axis='x', rotation=60)
axes[1].legend(fontsize=8)
axes[2].bar(effects.index.str.upper(), effects.values, color=['#117b8a', '#d89e36'])
axes[2].axhline(0, color='grey', lw=.8)
axes[2].set(ylabel='Macro-F1 decrease after shuffle', title='Held-out channel reliance')
fig.suptitle(EXPERIMENT_LABEL)
plt.tight_layout()
buffer = io.BytesIO()
fig.savefig(buffer, format='png', dpi=150, bbox_inches='tight')
plot_data = base64.b64encode(buffer.getvalue()).decode('ascii')
plt.show()
plt.close(fig)

target_met = summary['subject_macro_f1_mean'] > .8
strongest = effects.idxmax()
takeaways = [
    f"Subject-mean macro F1 is {summary['subject_macro_f1_mean']:.3f}; the >80% target was {'met' if target_met else 'not met'} in this experiment.",
    f"The final app model was refitted on all {len(folds)} usable participants. Reported scores belong to the LOSO procedure, not a new test of that final model.",
    (f"Shuffling {strongest.upper()} produced the larger mean macro-F1 decrease ({effects[strongest]:.3f}), suggesting greater model reliance in this diagnostic."
     if effects.max() > 0 else 'Neither channel showed a positive mean permutation effect; no clear input-reliance ranking was established.'),
    'The model learns waveform patterns directly. This experiment does not establish RMSSD, LF/HF, or any specific physiological biomarker.',
    'WESAD laboratory accuracy does not validate a different PPG/GSR device, uncontrolled activity, clinical diagnosis, or calibrated probability confidence.'
]
takeaways.insert(1, f"Observed-class macro F1 averages {summary['subject_macro_f1_observed_classes_mean']:.3f}. "
                 'This diagnostic excludes absent true classes; the primary score still uses all three classes and is not replaced by this larger number.')
takeaways.insert(2, f"Retained {int(totals.accepted.sum())}/{int(totals.pure_candidates.sum())} pure target windows; "
                 f"{int(totals.legacy_accepted.sum())} would pass the old strict beat gate. Compare retention and class support before comparing scores.")

report_payload = {'experiment': EXPERIMENT_LABEL, 'summary': summary, 'per_class': per_class,
                  'normalized_confusion_matrix': cm.tolist(), 'folds': fold_records,
                  'channel_permutation': sensor_effects, 'window_audit': ingestion_audit,
                  'alignment_audit': alignment_audit, 'skipped_subjects': errors,
                  'discovery': discovery.to_dict(orient='records'), 'final_tuning': final_trials}
report_payload.update(class_retention=class_retention_rows, eda_subject_class_summary=eda_cohort.to_dict(orient='records'),
                      historical_comparison=legacy_comparison)
config = {'format_version': 2, 'model_type': 'residual_1d_cnn_bilstm_attention',
          'class_names': CLASS_NAMES, 'original_label_map': LABEL_MAP,
          'input_shapes': {'bvp': [None, 1920, 1], 'eda': [None, 120, 4]},
          'input_units': {'bvp': 'Empatica E4 waveform amplitude (a.u.)', 'eda': 'microsiemens'},
          'preprocessing': PREPROCESSING,
          'normalization': 'window-robust BVP and four EDA channels in stress_inference.py; training-only channel normalization embedded in model; use bundled predictor without additional normalization',
          'trained_subjects': sorted(set(groups)), 'training_windows': len(y),
          'final_learning_rate': final_choice['learning_rate'], 'final_epochs': final_choice['epochs'],
          'use_absolute_eda_level': final_choice['absolute_eda'],
          'training_policy': {'balance': 'equal weight per observed subject/class cell', 'optimizer': 'AdamW',
                              'weight_decay': 1e-4, 'cosine_epochs': MAX_EPOCHS, 'early_start_epoch': EARLY_START_EPOCH,
                              'normalization_choices': EDA_CONTEXT_OPTIONS, 'outer_inner_splits': OUTER_INNER_SPLITS},
          'seed': SEED, 'versions': versions, 'experiment': EXPERIMENT_LABEL,
          'evaluation': 'LOSO procedure; final all-participant refit has no independent test score',
          'inference_contract': 'Raw BVP/EDA arrays from the same session origin, uniformly sampled at 64/4 Hz; insert NaN for missing samples; supply history from session start.',
          'runtime': 'Python TensorFlow backend; browser execution and TFLite conversion are not included'}
usage = '''from stress_inference import StressPredictor

predictor = StressPredictor("extracted_bundle_folder")
# Raw arrays from the same session origin: BVP 64 Hz, EDA 4 Hz.
# Insert NaN for dropped samples. Keep the session history for causal filtering.
result = predictor.predict_latest(bvp_session, eda_session)
# result: status, prediction, class_id, probabilities, window timestamps
'''
auc_text = f"{summary['subject_roc_auc_ovr_macro_mean']:.3f}" if valid_aucs else 'Not estimable'
report_html = f'''<!doctype html><html lang="en"><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Hybrid stress model — results and app guide</title>
<style>body{{font:16px/1.6 system-ui,sans-serif;max-width:1100px;margin:40px auto;padding:0 24px;color:#193046}}
h1,h2{{line-height:1.2}}.cards{{display:flex;gap:16px;flex-wrap:wrap}}.card{{background:#edf5f6;padding:18px;border-radius:10px;flex:1}}
.number{{font-size:30px;font-weight:700}}table{{border-collapse:collapse;width:100%;font-size:14px}}th,td{{padding:8px;border-bottom:1px solid #dde5e8;text-align:left}}
pre{{white-space:pre-wrap;overflow-wrap:anywhere;background:#f3f5f7;padding:16px;border-radius:8px}}img{{width:100%;height:auto}}small{{color:#536979}}</style>
<h1>Hybrid CNN–BiLSTM</h1><p>{html.escape(EXPERIMENT_LABEL)} · {len(folds)} participants · {len(y):,} accepted windows</p>
<div class="cards"><div class="card">Subject-mean macro F1<div class="number">{summary['subject_macro_f1_mean']:.3f}</div>
<small>Approximate 95% interval: {summary['subject_f1_ci95'][0]:.3f}–{summary['subject_f1_ci95'][1]:.3f}</small></div>
<div class="card">Subject-mean accuracy<div class="number">{summary['subject_accuracy_mean']:.3f}</div></div>
<div class="card">Subject-mean ROC-AUC<div class="number">{auc_text}</div></div></div>
<h2>Results</h2><img alt="LOSO confusion matrix, participant scores, and channel reliance" src="data:image/png;base64,{plot_data}">
<ul>{''.join('<li>' + html.escape(t) + '</li>' for t in takeaways)}</ul>
<h2>EDA and retention diagnosis</h2><img alt="EDA level, phasic variation and class retention" src="data:image/png;base64,{eda_plot_data}">
<p>The previous real run discarded 58.08% of pure target windows through a beat-quality gate, leaving 285 stress windows.
This version retains that check as an advisory for waveform classification. Missing/invalid samples and flat signals still fail quality screening.
Restored windows can contain motion-corrupted BVP: higher retention is not proof of higher signal quality or better accuracy.</p>
<p>EDA inputs contain log conductance level, locally robust relative log level, relative phasic residual and the derivative of log conductance.
The tonic component uses a causal exponential smoother and is an approximation. Inner participant validation chooses whether to include absolute EDA level.
No complete test recording or labeled baseline is used to fit normalization. Residual CNNs, a BiLSTM and attention pooling model the resulting waveforms.</p>
<details><summary>Historical comparison and old-gate subset</summary><pre>{html.escape(json.dumps(legacy_comparison, indent=2, allow_nan=False))}</pre></details>
<h2>Per-class results</h2>{pd.DataFrame({k: per_class[k] for k in CLASS_NAMES}).T.round(3).to_html()}
<h2>Participant results</h2>{folds[['subject','windows','classes_present','f1_macro','f1_macro_observed_classes','accuracy','roc_auc_ovr_macro']].round(3).to_html(index=False)}
<p>Primary macro F1 always includes all three classes. A participant with only one observed class can have perfect accuracy but a maximum primary macro F1 of 1/3.
The supported-class diagnostic explains this limitation without changing the presentation target.</p>
<h2>Use in a website or app backend</h2><p>Extract all four bundle files into one folder. Use Python, TensorFlow, NumPy and SciPy; tested versions are in model_config.json.
Load the model once in your backend process. A website/mobile client can call that backend; a network API and frontend are separate app work.</p>
<pre>{html.escape(usage)}</pre><p>The supplied inference module filters raw recordings and constructs robust channels using exactly the training code. The saved model then applies training-only channel normalization.
Prediction attempts occur every 5 seconds on a 30-second window. With 10 seconds of filter warmup, the earliest normal valid output is at 40 seconds; gaps and quality screening can delay it further.
A rejected window returns a status with no prediction. Do not reuse an old prediction as if it were current.</p>
<p>This reference implementation reprocesses the supplied session history on each call. It prioritizes training/inference consistency and requires optimization for long-running streams.
It expects a common origin and the specified sample rates; it does not align arbitrary BLE timestamps or resample a different device for you.</p>
<p>Version 2 uses a four-channel EDA representation. Keep the version 2 model, config and inference module together; the previous single-channel EDA input is incompatible.
An irregular pulse produces a quality_warning with the prediction. Your app should display that warning and avoid presenting probabilities as calibrated confidence.</p>
<h2>Method and limits</h2><p>Outer LOSO evaluates held-out participants. Participant validation inside each outer fold selects EDA context and stopping epoch.
Final deployment settings come from a separate group-CV search on the full cohort before an all-participant refit. The bootstrap interval resamples participant scores;
LOSO training overlap makes it approximate. Pooled windows overlap and are not independent observations. Channel permutation creates artificial paired inputs and is not a causal test.</p>
<p>This revision was motivated by the previous cohort's results. Although each fold keeps participants separate during fitting, rerunning this cohort is a development comparison; a fresh external cohort is needed for independent confirmation.</p>
<p>Pure protocol labels are used only offline. Live transition behavior, motion/contact robustness, probability calibration, hardware transfer and runtime latency require separate validation.
This is a research model, not a clinical diagnostic. No browser-native or TFLite model is included.</p>
<details><summary>Detailed experiment record and exclusions</summary><pre>{html.escape(json.dumps(report_payload, indent=2, allow_nan=False))}</pre></details>
<h2>Sources</h2><p><a href="https://archive.ics.uci.edu/dataset/465/wesad+wearable+stress+and+affect+detection">WESAD dataset</a> ·
<a href="https://www.eti.uni-siegen.de/ubicomp/papers/ubi_icmi2018.pdf">WESAD paper</a> ·
<a href="https://keras.io/api/layers/preprocessing_layers/numerical/normalization/">Keras normalization</a> ·
<a href="https://keras.io/api/optimizers/adamw/">AdamW</a> ·
<a href="https://keras.io/guides/serialization_and_saving/">Keras model serialization</a></p></html>'''


### Export exactly one download
Temporary staging is outside Kaggle working and is removed automatically. The ZIP
contains four files and no recordings, fold models, intermediate CSVs or loose PNGs.
Existing files from earlier notebook versions are left untouched; use a fresh Kaggle
run if you want a clean working directory. Open `results.html` after extracting.


In [ ]:
archive_path = OUTPUT / 'stress_app_bundle.zip'
with tempfile.TemporaryDirectory(prefix='wesad_hybrid_') as directory:
    staging = Path(directory)
    final_model.save(staging / 'stress_model.keras')
    restored = tf.keras.models.load_model(staging / 'stress_model.keras', compile=False)
    probe = inputs_for(np.arange(min(8, len(y))))
    np.testing.assert_allclose(predict_scores(final_model, probe), predict_scores(restored, probe), atol=1e-5, rtol=1e-5)
    config['model_sha256'] = hashlib.sha256((staging / 'stress_model.keras').read_bytes()).hexdigest()
    config['inference_source_sha256'] = hashlib.sha256(INFERENCE_SOURCE.encode('utf-8')).hexdigest()
    (staging / 'model_config.json').write_text(json.dumps(config, indent=2, allow_nan=False), encoding='utf-8')
    (staging / 'stress_inference.py').write_text(INFERENCE_SOURCE, encoding='utf-8', newline='\n')
    (staging / 'results.html').write_text(report_html, encoding='utf-8')
    # Write atomically so an interrupted export cannot replace a valid earlier bundle.
    temporary_archive = OUTPUT / 'stress_app_bundle.zip.tmp'
    try:
        with zipfile.ZipFile(temporary_archive, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
            for name in ['stress_model.keras', 'model_config.json', 'stress_inference.py', 'results.html']:
                archive.write(staging / name, arcname=name)
        temporary_archive.replace(archive_path)
    finally:
        temporary_archive.unlink(missing_ok=True)
print('Download only:', archive_path)
print('Inside: stress_model.keras | model_config.json | stress_inference.py | results.html')
print('Saved-model reload check passed. Actual conclusions are in results.html.')
